# S15 · ¿Cómo recuperar un pipeline tras un fallo?
**Universidad Central · Big Data · propuesta formativa sin nota**

Laura recibe registros de contratación en lotes. Si falla la segunda descarga, no debe perder datos ni repetir registros. Construiremos un pipeline pequeño que permite probar la idea sin contratar servicios. Los datos son sintéticos, con errores intencionales; no son registros SECOP observados.

**Ruta:** preparar → procesar → provocar un fallo → recuperar → verificar → transferir. **Producto:** manifiesto de ejecución y evidencias de idempotencia, rechazo y recuperación.

En esta práctica SQLite ilustra transacciones y estado duradero en un proceso local. Después se relaciona con Delta y Structured Streaming, pero no se presenta como un clúster ni como una prueba de tolerancia distribuida.

## 1 · Preparar el contrato y los lotes
Una clave identifica el evento. El lote se confirma después de escribir los datos y su checkpoint en la misma transacción. Un rechazo conserva el evento y la causa; no se convierte un dato inválido en cero.

**HAZ ESTO AHORA.** Ejecuta la celda; observarás tres lotes y un archivo de trabajo. No requiere instalaciones.

In [ ]:
import sqlite3, json, tempfile
from pathlib import Path
from hashlib import sha256

carpeta = Path(tempfile.mkdtemp(prefix="s15_"))
ruta = carpeta / "pipeline.db"
lotes = [
    [{"id":"P1","valor":100},{"id":"P2","valor":200},{"id":"P2","valor":200}],
    [{"id":"P3","valor":300},{"id":"P4","valor":None},{"id":"P5","valor":500}],
    [{"id":"P6","valor":600,"campo_nuevo":"v2"},{"id":"P7","valor":700}]
]
with sqlite3.connect(ruta) as db:
    db.executescript("""
    CREATE TABLE validos(id TEXT PRIMARY KEY, valor INTEGER NOT NULL);
    CREATE TABLE rechazos(id TEXT PRIMARY KEY, causa TEXT NOT NULL, raw TEXT NOT NULL);
    CREATE TABLE lotes_confirmados(lote INTEGER PRIMARY KEY, hash TEXT NOT NULL);
    """)
print("Lotes:", len(lotes), "| archivo:", ruta.name)

**Cómo se lee:** cada lote contiene registros y el archivo guarda las tres tablas. **Qué nos dice:** ya tenemos un contrato y un estado persistente. **Qué NO permite concluir:** todavía no se procesó ningún lote. **Qué error común:** usar el número de fila como identidad estable.

## 2 · Procesar de forma incremental
El código que importa está visible: valida registros, escribe por identidad y confirma un checkpoint. Si una excepción sale del contexto de SQLite, se revierte la transacción completa. El hash de cada lote impide aceptar silenciosamente cambios en un lote confirmado. Una clave repetida con otro valor se rechaza como conflicto, no se sobrescribe sin política.

In [ ]:
def huella(registros):
    return sha256(json.dumps(registros, sort_keys=True, separators=(",",":")).encode()).hexdigest()

def estado():
    with sqlite3.connect(ruta) as db:
        return {t: db.execute("SELECT COUNT(*) FROM "+t).fetchone()[0]
                for t in ("validos","rechazos","lotes_confirmados")}

def procesar(numero, registros, fallar=False):
    h = huella(registros)
    with sqlite3.connect(ruta) as db:
        db.execute("BEGIN IMMEDIATE")
        anterior = db.execute("SELECT hash FROM lotes_confirmados WHERE lote=?", (numero,)).fetchone()
        if anterior:
            if anterior[0] != h:
                raise ValueError("El lote confirmado cambió de contenido")
            return "YA CONFIRMADO"
        for pos, r in enumerate(registros):
            ident = r.get("id")
            valor = r.get("valor")
            raw = json.dumps(r, sort_keys=True, ensure_ascii=False)
            clave_rechazo = f"{numero}:{pos}"
            if not isinstance(ident, str) or not ident or type(valor) is not int or valor < 0:
                db.execute("INSERT INTO rechazos VALUES(?,?,?)",
                           (clave_rechazo,"contrato invalido",raw))
            else:
                previa = db.execute("SELECT valor FROM validos WHERE id=?", (ident,)).fetchone()
                if previa is None:
                    db.execute("INSERT INTO validos VALUES(?,?)", (ident,valor))
                elif previa[0] != valor:
                    db.execute("INSERT INTO rechazos VALUES(?,?,?)",
                               (clave_rechazo,"conflicto de identidad",raw))
            if fallar and pos == 0:
                raise RuntimeError("FALLO CONTROLADO")
        db.execute("INSERT INTO lotes_confirmados VALUES(?,?)", (numero,h))
    return "CONFIRMADO"

print(procesar(0,lotes[0]), estado())
assert estado() == {"validos":2,"rechazos":0,"lotes_confirmados":1}

**Cómo se lee:** 2 válidos, 0 rechazos y 1 lote confirmado. **Qué nos dice:** el duplicado idéntico P2 no crea una segunda fila. **Qué NO permite concluir:** aún no se ha probado la recuperación. **Qué error común:** actualizar el checkpoint antes de persistir los datos.

## 3 · Fallo controlado y recuperación
Predice: si el segundo lote falla después de escribir P3, ¿debería aparecer P3 al abrir otra conexión? Ejecuta y comprueba. No se cae un servicio externo.

In [ ]:
antes = estado()
try:
    procesar(1,lotes[1],fallar=True)
except RuntimeError as exc:
    assert str(exc) == "FALLO CONTROLADO"
else:
    raise AssertionError("No se provocó el fallo")
despues = estado()
assert antes == despues
print("Rollback comprobado:", despues)
print(procesar(1,lotes[1]), estado())
assert estado() == {"validos":4,"rechazos":1,"lotes_confirmados":2}

**Cómo se lee:** el primer estado no cambia tras el fallo; el reintento produce 4 válidos, 1 rechazo, 2 lotes. **Qué nos dice:** la transacción revierte la escritura parcial y permite reintentar. **Qué NO permite concluir:** la recuperación de un executor distribuido no fue probada. **Qué error común:** confundir reintento con repetir una inserción sin identidad.

## 4 · Repetición, cambio de esquema y evidencia
El campo nuevo no altera las columnas que requiere el contrato. El RAW sintético conserva ese campo en el lote original, pero no lo añade automáticamente a la tabla Silver. Aquí se conserva todo RAW en el manifiesto; en producción se almacenaría por separado.

**HAZ ESTO AHORA.** Repite un lote confirmado, ejecuta el tercero y prueba que un lote confirmado con contenido modificado se rechaza.

In [ ]:
antes_repetir = estado()
assert procesar(1,lotes[1]) == "YA CONFIRMADO"
assert estado() == antes_repetir
print(procesar(2,lotes[2]), estado())
assert estado() == {"validos":6,"rechazos":1,"lotes_confirmados":3}
try:
    procesar(2,[{"id":"P6","valor":999}])
except ValueError as exc:
    print("Contrato de lote:", exc)
else:
    raise AssertionError("Se aceptó un lote modificado")
with sqlite3.connect(ruta) as db:
    filas = db.execute("SELECT id,valor FROM validos ORDER BY id").fetchall()
    rechazos = db.execute("SELECT id,causa,raw FROM rechazos").fetchall()
    confirmados = db.execute("SELECT lote,hash FROM lotes_confirmados ORDER BY lote").fetchall()
manifest = {"fuente":"sintetica", "estado":estado(), "validos":filas,
            "rechazos":rechazos, "checkpoints":confirmados, "raw":lotes,
            "rollback_verificado":antes==despues}
salida = carpeta / "manifiesto_s15.json"
salida.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print("Evidencia:", salida)
print(filas, rechazos)

**Cómo se lee:** 6 registros válidos y 1 rechazo, con 3 lotes confirmados. **Qué nos dice:** la ejecución repetida no duplica resultados; la identidad y el checkpoint protegen el estado. **Qué NO permite concluir:** no se ha evaluado throughput, seguridad multiusuario ni recuperación distribuida. **Qué error común:** llamar exactly-once a cualquier reintento; depende de fuente, destino y garantías de escritura.

## 5 · Reto de transferencia (formativo)
Agrega un cuarto lote que contenga un identificador existente con valor diferente y un registro nuevo válido. Predice qué conteos cambiarán, modifica solo el valor indicado y ejecuta la celda. No hace falta escribir una explicación libre.

In [ ]:
nuevo_valor = 999  # modifica: 100 repite P1; otro entero positivo entra en conflicto
lote4 = [{"id":"P1","valor":nuevo_valor},{"id":"P8","valor":800}]
inicio = estado()
print(procesar(3,lote4), estado())
fin = estado()
assert fin["validos"] == inicio["validos"] + 1
assert fin["rechazos"] == inicio["rechazos"] + int(nuevo_valor != 100)
assert fin["lotes_confirmados"] == inicio["lotes_confirmados"] + 1
assert procesar(3,lote4) == "YA CONFIRMADO"
assert estado() == fin
print("Transferencia verificada")

Si deseas volver a probar otro valor, reinicia desde el bloque 1: cada lote confirmado es inmutable. **PARA LLEVAR:** un checkpoint solo es confiable si coincide con la escritura confirmada. En Spark Structured Streaming hay checkpoints y gestión de microbatches, pero la semántica depende del origen y destino. Lee la [documentación de Spark Structured Streaming](https://spark.apache.org/docs/latest/streaming/index.html) como ampliación; no exige instalar otra plataforma. El proyecto final conserva su alcance vigente.

## 6 · Ampliación opcional: microbatches reales con Spark Structured Streaming
**Solo si ya tienes PySpark clásico local disponible (S14), con Java y Spark 3.3 o superior.** No instala bibliotecas, no se ejecuta automáticamente y no usa Databricks Serverless, porque esta práctica requiere rutas locales y Spark clásico.

Vamos a escribir dos pequeños archivos JSON de eventos, ejecutar una consulta de streaming acotada (`availableNow`), detenerla y reanudar con el mismo checkpoint. El destino Parquet y el origen de archivos son parte de esta prueba concreta; no afirmamos garantías universales de exactly-once.

**HAZ ESTO AHORA.** Cambia `EJECUTAR_SPARK` solo si has preparado Spark local y dispones de recursos. Antes de ejecutar, predice si el primer archivo se volverá a procesar al reiniciar la consulta.

In [ ]:
EJECUTAR_SPARK = False  # ampliación voluntaria, sin instalación ni red
if EJECUTAR_SPARK:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import StructType, StructField, StringType, LongType
    import shutil
    spark_local = SparkSession.getActiveSession()
    creado = spark_local is None
    if creado:
        spark_local = SparkSession.builder.master("local[2]").appName("S15-microbatch").getOrCreate()
    if not spark_local.sparkContext.master.startswith("local"):
        raise RuntimeError("Utiliza Spark clásico en modo local, no un servidor remoto")
    base_stream = Path(tempfile.mkdtemp(prefix="s15_stream_"))
    origen = base_stream / "origen"
    origen.mkdir()
    destino = str(base_stream / "parquet")
    checkpoint = str(base_stream / "checkpoint")
    schema = StructType([StructField("id",StringType(),False),
                         StructField("valor",LongType(),False)])

    def publicar_archivo(nombre, eventos):
        borrador = origen / ("."+nombre+".part")
        borrador.write_text("\n".join(json.dumps(x) for x in eventos)+"\n")
        borrador.rename(origen / nombre)

    def ejecutar_microbatch():
        eventos = (spark_local.readStream.schema(schema)
                   .option("maxFilesPerTrigger",1).json(str(origen)))
        q = (eventos.writeStream.format("parquet").outputMode("append")
             .option("path",destino).option("checkpointLocation",checkpoint)
             .trigger(availableNow=True).start())
        try:
            if not q.awaitTermination(90):
                raise TimeoutError("No terminó el microbatch dentro del límite")
            if q.exception() is not None:
                raise RuntimeError(str(q.exception()))
        finally:
            if q.isActive:
                q.stop()
        return sorted(r.id for r in spark_local.read.parquet(destino).select("id").collect())

    try:
        publicar_archivo("lote1.json", [{"id":"S1","valor":10},{"id":"S2","valor":20}])
        primero = ejecutar_microbatch()
        assert primero == ["S1","S2"]
        publicar_archivo("lote2.json", [{"id":"S3","valor":30}])
        segundo = ejecutar_microbatch()
        assert segundo == ["S1","S2","S3"]
        tercero = ejecutar_microbatch()
        assert tercero == segundo
        print("Primero:", primero, "| Recuperado:", segundo, "| Repetición:", tercero)
    finally:
        if creado:
            spark_local.stop()
        shutil.rmtree(base_stream, ignore_errors=True)
else:
    print("Ampliación Spark no ejecutada; la ruta esencial ya está completa.")

**Cómo se lee:** si ejecutas la ampliación, aparecen 2, luego 3 y finalmente 3 identidades. **Qué nos dice:** el checkpoint permite reanudar esta consulta con el mismo origen y destino. **Qué NO permite concluir:** no se simularon fallos de nodos, pérdida de almacenamiento ni concurrencia entre productores. **Qué error común:** borrar el checkpoint y esperar que se mantenga el historial. El laboratorio SQLite anterior muestra de manera independiente rollback atómico y conflictos de identidad.

## Laboratorio ampliado · Fallo inyectado, replay y equivalencia de estados

### Un problema de recuperación, no de sintaxis
Un sistema recibe eventos A v1, B v1, A v2 y C v1. Un checkpoint indica la siguiente posición. Si la escritura y el checkpoint no se confirman atómicamente, un fallo entre ambos puede producir replay o pérdida. La estrategia append conserva todos los intentos, mientras upsert por identidad y versión conserva el estado más reciente. Repetir un evento no es lo mismo que aplicar una corrección de versión posterior.

La práctica usa SQLite real con archivos independientes en un directorio temporal. Simula un fallo después de escribir y antes de confirmar posición, o después de adelantar posición y antes de escribir. Cierra y abre otra conexión para recuperar estado; no utiliza variables como único mecanismo de persistencia. Ejecuta una referencia limpia con la misma política y compara filas ordenadas. En append se conserva multiplicidad, en upsert la última versión. El caso ilustrativo permite comprobar idempotencia y pérdida de eventos de manera reproducible.

### Qué observar
Con checkpoint posterior y upsert, se repite el evento pero el estado materializado puede coincidir. Con append, el mismo fallo deja una fila adicional. Con checkpoint adelantado, el evento puede no recuperarse: si es sustituido por una versión posterior, el estado final puede coincidir aun cuando faltó un evento; por eso también se conservan trazas y se distingue estado final de historia completa. La práctica no demuestra garantías de un sistema distribuido.

**Reto:** mueve el fallo a posición uno y explica por qué perder A v1 podría no alterar el estado final si después llega A v2. Después mueve el fallo a B v1: ¿qué identidad desaparece? Compara con el laboratorio transaccional original.


In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory
eventos_s15=[("A",1),("B",1),("A",2),("C",1)]
def replay_s15(ruta,modo,fallo=0,adelantado=False):
    with closing(sqlite3.connect(ruta)) as cx, cx:
        cx.execute("CREATE TABLE estado(id TEXT, version INTEGER"+(", PRIMARY KEY(id)" if modo=="upsert" else "")+")")
        cx.execute("CREATE TABLE checkpoint(pos INTEGER)")
        cx.execute("INSERT INTO checkpoint VALUES(0)")
    traza=[]
    def escribir(cx,e):
        if modo=="upsert":
            cx.execute("INSERT INTO estado VALUES(?,?) ON CONFLICT(id) DO UPDATE SET version=excluded.version WHERE excluded.version>=estado.version",e)
        else: cx.execute("INSERT INTO estado VALUES(?,?)",e)
    def avanzar(cx,p): cx.execute("UPDATE checkpoint SET pos=?",(p,))
    inyectado=False
    while True:
        with closing(sqlite3.connect(ruta)) as cx, cx:
            p=cx.execute("SELECT pos FROM checkpoint").fetchone()[0]
            if p==len(eventos_s15): break
            e=eventos_s15[p]
            if p+1==fallo and not inyectado:
                if adelantado: avanzar(cx,p+1)
                else: escribir(cx,e)
                traza.append((p,"fallo"))
                inyectado=True
            else:
                escribir(cx,e)
                avanzar(cx,p+1)
                traza.append((p,"confirmado"))
    with closing(sqlite3.connect(ruta)) as cx, cx:
        estado=cx.execute("SELECT id,version FROM estado ORDER BY id,version").fetchall()
    return estado,traza
with TemporaryDirectory(prefix="s15_revision_") as tmp:
    for modo in ("append","upsert"):
        base,_=replay_s15(Path(tmp)/(modo+"_base.db"),modo)
        recuperado,traza=replay_s15(Path(tmp)/(modo+"_replay.db"),modo,2,False)
        perdido,_=replay_s15(Path(tmp)/(modo+"_perdida.db"),modo,2,True)
        print(modo,"limpio",base,"replay",recuperado,"adelantado",perdido,"traza",traza)
        assert (base==recuperado)==(modo=="upsert")
        assert not any(i=="B" for i,_ in perdido)
